<a href="https://colab.research.google.com/github/craljimenez/An-lisis_y_Visualizacion_de_Datos_para_la_Toma_de_Decisiones/blob/main/clases/semana06-clase11-principios-visualizacion-comunicacion-datos/notebooks/lab-visualizacion-comunicacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio: Principios de visualización y comunicación de datos
### Semana 6 · Clase 11
**Curso:** Análisis y Visualización de Datos para la Toma de Decisiones (Esumer)

En la Clase 9 agrupaste los datos de los socios y en la Clase 10 los integraste con otras fuentes. Hoy das el paso siguiente: **mostrar** esos resultados de forma que alguien que no conoce los datos entienda el mensaje en segundos.

Vas a construir cada tipo de gráfico de las diapositivas usando, sobre todo, el método `.plot()` que ya conoces de pandas. Al final hay un **taller**: tres gráficos deliberadamente mal hechos que vas a rediseñar y justificar.

**Sobre los enlaces de este notebook:** la documentación oficial de pandas y matplotlib existe solo en inglés. Para la explicación conceptual de cada tipo de gráfico en español, cada sección enlaza también a [Data Viz Catalogue](https://datavizcatalogue.com/ES/), el mismo catálogo visual que viste en las diapositivas.

In [ ]:
from google.colab import files

uploaded = files.upload()

Selecciona `membresias_gimnasio_limpio.csv`, de la carpeta `datos/` de esta clase en el repositorio del curso.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

socios = pd.read_csv("membresias_gimnasio_limpio.csv")
socios["fecha_inscripcion"] = pd.to_datetime(socios["fecha_inscripcion"])
socios.shape

Define la paleta de colores que vas a usar en todo el notebook (los mismos colores de las diapositivas), para que cada figura se vea consistente con la anterior.

In [ ]:
TEAL = "#0CB3B6"
NAVY = "#262E66"
ORANGE = "#EF7C2C"
SKY = "#26A1DB"
MAGENTA = "#AF348B"
GREEN = "#94C122"

## 1. Comparación

**Pregunta:** ¿cómo se compara algo entre categorías, o a lo largo del tiempo?

Con pocas categorías en un momento dado, usa barras. Para la evolución en el tiempo, usa una línea. Las dos salen directo de `.plot()`, el mismo método que ya usaste para graficar una `Series` en clases anteriores.

### 1.1 Pocas categorías: barras horizontales

Ordena antes de graficar (no dejes el orden en que llegaron los datos) y pon la conclusión en el título, no una descripción genérica.

In [ ]:
por_ciudad = socios["ciudad"].value_counts().sort_values()

por_ciudad.plot.barh(color=TEAL, title=f"{por_ciudad.idxmax()} concentra la mayor parte de los socios")
plt.xlabel("Socios")
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`Series.plot.barh`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.barh.html)
- Guía conceptual (en español): [Gráficos de Barras, Data Viz Catalogue](https://datavizcatalogue.com/ES/metodos/graficos_de_barras.html)

### 1.2 Evolución en el tiempo: línea

Agrupa por mes con `dt.to_period("M")`, igual que en la Clase 9, y grafica directamente el resultado.

In [ ]:
por_mes = socios.groupby(socios["fecha_inscripcion"].dt.to_period("M")).size()
por_mes.index = por_mes.index.strftime("%b %y")

por_mes.plot(marker="o", color=TEAL, title="Inscripciones por mes")
plt.ylabel("Inscripciones")
plt.xticks(rotation=45)
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`Series.plot.line`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.line.html)
- Guía conceptual (en español): [Gráfica de Línea, Data Viz Catalogue](https://datavizcatalogue.com/ES/metodos/grafica_de_linea.html)

Regla práctica de las diapositivas: con más de 10 a 12 categorías, una tabla ordenada suele comunicar mejor que un gráfico de barras saturado.

## 2. Composición

**Pregunta:** ¿cómo se reparte un total en sus partes?

Barras apiladas (o apiladas al 100%) cuando la composición cambia entre grupos; pastel **solo** con pocas categorías (3 a 5) y diferencias claras entre ellas.

### 2.1 Composición que cambia: barras apiladas al 100%

Participación de cada plan dentro de cada ciudad (como en la Clase 9, con `.transform("sum")`; aquí lo haces con `pd.crosstab`).

In [ ]:
tabla = pd.crosstab(socios["ciudad"], socios["plan"])
tabla_pct = tabla.div(tabla.sum(axis=1), axis=0) * 100

tabla_pct.plot.bar(stacked=True, color=[TEAL, SKY, ORANGE],
                    title="Participación de cada plan dentro de cada ciudad")
plt.ylabel("Porcentaje de socios")
plt.xticks(rotation=0)
plt.legend(title="Plan", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`DataFrame.plot.bar`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.bar.html) (parámetro `stacked=True`)
- Guía conceptual (en español): [Gráfico de Barras Apiladas, Data Viz Catalogue](https://datavizcatalogue.com/ES/metodos/grafico_de_barras_apiladas.html)

### 2.2 Una sola composición: pastel (solo con pocas categorías)

Aquí hay 3 planes, con diferencias claras: es uno de los pocos casos donde un pastel es razonable.

In [ ]:
por_plan = socios["plan"].value_counts()

por_plan.plot.pie(autopct="%1.0f%%", colors=[TEAL, SKY, ORANGE], ylabel="", title="Socios por plan")
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`Series.plot.pie`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.pie.html)
- Guía conceptual (en español): [Gráficos de Tarta, Data Viz Catalogue](https://datavizcatalogue.com/ES/metodos/graficos_de_tarta.html)

Más adelante en este notebook (Sección 8) vas a ver qué pasa cuando un pastel tiene demasiadas categorías.

## 3. Distribución

**Pregunta:** ¿cómo se distribuyen los valores de una variable? ¿Dónde se concentran, qué tan dispersos están, hay valores atípicos?

### 3.1 Una variable: histograma

Agrupa los valores en rangos (`bins`) y cuenta cuántas observaciones caen en cada uno (como en la Clase 2).

In [ ]:
socios["edad"].plot.hist(bins=10, color=TEAL, edgecolor="white", title="Distribución de edades")
plt.xlabel("Edad")
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`Series.plot.hist`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.hist.html)
- Guía conceptual (en español): [Histograma, Data Viz Catalogue](https://datavizcatalogue.com/ES/metodos/histograma.html)

### 3.2 Comparar distribuciones: boxplot por grupo

Resume mediana, cuartiles y valores atípicos de cada grupo; útil para comparar varios a la vez (como en la Clase 2). `DataFrame.boxplot` con `by=` arma la comparación sin que tengas que separar los grupos tú mismo.

In [ ]:
socios.boxplot(column="edad", by="plan")
plt.title("Distribución de edad por plan")
plt.suptitle("")
plt.ylabel("Edad")
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`DataFrame.boxplot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.boxplot.html)
- Guía conceptual (en español): [Diagrama Cajas y Bigotes, Data Viz Catalogue](https://datavizcatalogue.com/ES/metodos/diagrama_cajas_y_bigotes.html)

## 4. Relación

**Pregunta:** ¿cómo se relacionan dos variables entre sí? Cada punto es una observación, un eje por variable.

In [ ]:
r = socios["edad"].corr(socios["antiguedad_dias"])

socios.plot.scatter(x="edad", y="antiguedad_dias", color=TEAL, title=f"Edad vs. antigüedad (r = {r:.2f})")
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`DataFrame.plot.scatter`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.scatter.html)
- Guía conceptual (en español): [Diagrama de Dispersión, Data Viz Catalogue](https://datavizcatalogue.com/ES/metodos/diagrama_de_dispersion.html)

La correlación es prácticamente nula ($r \approx -0{,}09$). El gráfico de dispersión confirma lo que dice el número: no hay una relación lineal visible entre la edad y la antigüedad. Recuerda la advertencia de la Clase 2: un $r$ cercano a 0 descarta una relación **lineal**, no cualquier relación.

## 5. Escalas: el eje que decide lo que ves

Para **barras**, el eje de magnitud debe empezar en 0: el ojo compara el *tamaño* de las barras. Compara la misma información con dos rangos de eje distintos, una al lado de la otra con `plt.subplot`: le dices a matplotlib "divide la figura en 1 fila y 2 columnas, y dibuja aquí en la posición 1" (o 2).

In [ ]:
top3 = socios["ciudad"].value_counts().head(3)

plt.figure(figsize=(9, 4))

plt.subplot(1, 2, 1)
plt.bar(top3.index, top3.values, color=ORANGE)
plt.ylim(top3.min() - 2, top3.max() + 1)
plt.title("Truncado (engañoso)")

plt.subplot(1, 2, 2)
plt.bar(top3.index, top3.values, color=TEAL)
plt.ylim(0, top3.max() + 2)
plt.title("Honesto (empieza en 0)")

plt.suptitle("Mismos datos: la diferencia entre ciudades se ve mucho más grande a la izquierda")
plt.tight_layout()
plt.show()

Para **líneas** que muestran una tendencia, truncar el eje es aceptable si lo dices explícitamente. El riesgo es el mismo: exagerar visualmente un cambio pequeño.

In [ ]:
plt.figure(figsize=(9, 4))

plt.subplot(1, 2, 1)
plt.plot(por_mes.index, por_mes.values, color=ORANGE, marker="o")
plt.ylim(por_mes.min(), por_mes.max())
plt.title("Truncado")
plt.xticks(rotation=45, ha="right")

plt.subplot(1, 2, 2)
plt.plot(por_mes.index, por_mes.values, color=TEAL, marker="o")
plt.ylim(0, por_mes.max() + 1)
plt.title("Honesto")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`matplotlib.pyplot.ylim`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.ylim.html) · [`matplotlib.pyplot.subplot`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplot.html)
- Guía conceptual (en español): no existe traducción oficial de este principio; el capítulo de referencia es [Fundamentals of Data Visualization, cap. Visualizing amounts](https://clauswilke.com/dataviz/visualizing-amounts.html) (en inglés).

## 6. Color: con propósito, no por decoración

Para esta sección vamos a usar también **seaborn**, una librería que ya viene instalada en Colab y que genera paletas de color y gráficos estadísticos con menos código que matplotlib puro.

Tres formas de usar el color, con el mismo dato (socios por ciudad): categórico (identidad), secuencial (magnitud) y divergente (por encima/por debajo de un punto de referencia).

In [ ]:
import seaborn as sns

por_ciudad2 = socios["ciudad"].value_counts()
promedio = por_ciudad2.mean()

plt.figure(figsize=(13, 4.2))

plt.subplot(1, 3, 1)
paleta_cat = [TEAL, SKY, ORANGE, MAGENTA, GREEN]
plt.bar(por_ciudad2.index, por_ciudad2.values, color=paleta_cat[: len(por_ciudad2)])
plt.title("Categórico: una ciudad = un color")
plt.xticks(rotation=30, ha="right")

plt.subplot(1, 3, 2)
orden = por_ciudad2.sort_values()
colores_secuencial = sns.color_palette("Blues", len(orden))
plt.bar(orden.index, orden.values, color=colores_secuencial)
plt.title("Secuencial: color = cuánto")
plt.xticks(rotation=30, ha="right")

plt.subplot(1, 3, 3)
colores_divergente = [GREEN if v >= promedio else MAGENTA for v in por_ciudad2.values]
plt.bar(por_ciudad2.index, por_ciudad2.values, color=colores_divergente)
plt.axhline(promedio, color=NAVY, linestyle="--", linewidth=1)
plt.title("Divergente: sobre/bajo el promedio")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`seaborn.color_palette`](https://seaborn.pydata.org/generated/seaborn.color_palette.html) · [Choosing Colormaps in Matplotlib](https://matplotlib.org/stable/users/explain/colors/colormaps.html)
- Guía conceptual (en español): no hay traducción oficial sobre paletas y percepción del color; el capítulo de referencia es [Fundamentals of Data Visualization, cap. Color scales](https://clauswilke.com/dataviz/color-basics.html) (en inglés).

Máximo 5 a 6 colores categóricos: más que eso, nadie los distingue. Y revisa que la paleta se distinga también para quienes tienen daltonismo (evita depender solo de rojo y verde).

## 7. Doble eje Y: dos escalas, una sola impresión

Dos series reales, prácticamente sin relación entre sí: inscripciones por mes y edad promedio por mes. Con `secondary_y`, pandas pone la segunda columna en un eje derecho, sin que tengas que crear nada aparte.

In [ ]:
tabla2 = socios.groupby(socios["fecha_inscripcion"].dt.to_period("M")).agg(
    inscripciones=("id_socio", "count"), edad_prom=("edad", "mean")
)
tabla2.index = tabla2.index.strftime("%b %y")
r2 = tabla2["inscripciones"].corr(tabla2["edad_prom"])
print(f"correlación real entre las dos series: {r2:.2f}")

tabla2.plot(secondary_y="edad_prom", style=["o-", "s-"], color=[TEAL, ORANGE],
            title=f"Doble eje: parecen moverse juntas (r real = {r2:.2f})")
plt.xticks(rotation=45)
plt.show()

Pandas ajusta cada eje, en automático, al rango de **su propia** columna. El resultado: las dos líneas ocupan toda la altura del gráfico sin importar si tienen algo que ver. La alternativa más honesta es mostrar cada serie en su propio gráfico, una debajo de la otra:

In [ ]:
tabla2["inscripciones"].plot(marker="o", color=TEAL, title="Inscripciones por mes")
plt.xticks(rotation=45)
plt.show()

In [ ]:
tabla2["edad_prom"].plot(marker="s", color=ORANGE, title="Edad promedio por mes")
plt.xticks(rotation=45)
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`DataFrame.plot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html) (parámetro `secondary_y`)
- No hay un tipo de gráfico específico en Data Viz Catalogue para "doble eje"; es una variación de [Gráfica de Línea](https://datavizcatalogue.com/ES/metodos/grafica_de_linea.html) que conviene usar con cuidado.

## 8. Reducir el ruido visual

*Data-ink ratio* (Tufte): la proporción del gráfico dedicada a mostrar datos, frente a todo lo demás. Mismo dato, con y sin ruido:

In [ ]:
plt.figure(figsize=(10, 4.5))

plt.subplot(1, 2, 1)
colores_sin_proposito = ["#FF0000", "#00FF00", "#0000FF", "#FFFF00"][: len(por_ciudad2)]
plt.bar(por_ciudad2.index, por_ciudad2.values, color=colores_sin_proposito)
plt.grid(True, linewidth=1.5)
plt.legend(["Socios"])
plt.title("Con ruido")
plt.xticks(rotation=30, ha="right")

plt.subplot(1, 2, 2)
barras = plt.bar(por_ciudad2.index, por_ciudad2.values, color=TEAL)
plt.bar_label(barras)
plt.title("Sin ruido")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()

A la derecha: sin rejilla pesada, sin leyenda (la etiqueta ya está sobre la barra), un solo color con propósito (identifica la serie, no decora). El mensaje es el mismo; leerlo cuesta menos esfuerzo.

**Para profundizar:**
- Documentación oficial (en inglés): [`matplotlib.pyplot.bar_label`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar_label.html)

## 9. Dos gráficos más, para tenerlos a mano

No cubren una pregunta nueva, pero son útiles cuando el boxplot no es suficiente. Con seaborn, pasas directamente el nombre de las columnas (`x=`, `y=`), sin tener que separar los grupos tú mismo.

### 9.1 Violín: la forma completa de la distribución

In [ ]:
sns.violinplot(data=socios, x="plan", y="edad", order=por_plan.index, color=TEAL)
plt.title("Forma completa de la distribución de edad por plan")
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`seaborn.violinplot`](https://seaborn.pydata.org/generated/seaborn.violinplot.html)
- Guía conceptual (en español): [Diagrama de Violín, Data Viz Catalogue](https://datavizcatalogue.com/ES/metodos/diagrama_de_violin.html)

### 9.2 Barras con barras de error: un promedio y su incertidumbre

`seaborn.barplot` calcula el promedio de cada grupo y dibuja encima su intervalo de confianza, sin que tengas que calcular nada a mano.

In [ ]:
sns.barplot(data=socios, x="plan", y="edad", order=por_plan.index, color=TEAL)
plt.ylabel("Edad promedio (con intervalo de confianza)")
plt.title("Edad promedio por plan, con su variabilidad")
plt.show()

**Para profundizar:**
- Documentación oficial (en inglés): [`seaborn.barplot`](https://seaborn.pydata.org/generated/seaborn.barplot.html) · [`matplotlib.pyplot.errorbar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.errorbar.html) (la versión sin seaborn)
- Guía conceptual (en español): [Barras de Error, Data Viz Catalogue](https://datavizcatalogue.com/ES/metodos/barras_de_error.html)

Otros tipos que no construimos hoy, pero que vale la pena conocer: mapa de calor ([`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) · [Mapa de Calor, ES](https://datavizcatalogue.com/ES/metodos/mapa_de_calor.html)), área ([`matplotlib.pyplot.fill_between`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.fill_between.html) · [Gráfica de Área, ES](https://datavizcatalogue.com/ES/metodos/grafica_de_area.html)), burbujas ([`DataFrame.plot.scatter`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.scatter.html) con el parámetro `s=` como tercera variable · [Gráfico de Burbujas, ES](https://datavizcatalogue.com/ES/metodos/grafico_de_burbujas.html)) y treemap ([Plotly Treemaps](https://plotly.com/python/treemaps/) · [Mapa de Árbol, ES](https://datavizcatalogue.com/ES/metodos/mapa_de_arbol.html)).

## 10. Taller: rediseña tres gráficos deficientes

Los tres gráficos de abajo están construidos con los mismos datos reales que usaste en todo el notebook, pero cada uno viola al menos uno de los principios de hoy. Tu entregable:

1. Para cada gráfico, identifica **qué principio viola** y **por qué engaña o dificulta la lectura**.
2. Rediseña el gráfico. Puedes hacerlo en Python, en la celda `# TODO` correspondiente, **o si prefieres, en Excel**: exporta los datos con `socios.to_csv("socios.csv")` (lo descargas con el panel de archivos de Colab), arma el gráfico rediseñado allá, y pega aquí una captura de pantalla (arrastra la imagen a una celda de texto en Colab para insertarla). Lo importante es la decisión de diseño, no la herramienta.
3. En la celda de reflexión, justifica cada decisión de rediseño con lo visto hoy (no basta con "se ve mejor": explica qué principio aplicaste).

### Gráfico deficiente 1

In [ ]:
colores_random = ["#FF0000", "#00FF00", "#0000FF", "#FFFF00"]
conteo_ciudad = socios["ciudad"].value_counts()

plt.bar(conteo_ciudad.index, conteo_ciudad.values, color=colores_random[: len(conteo_ciudad)])
plt.ylim(15, 24)
plt.show()

In [ ]:
# TODO: rediseña el gráfico deficiente 1 (o pega aquí la captura de Excel)


### Gráfico deficiente 2

In [ ]:
combo = (socios["ciudad"] + " - " + socios["plan"]).value_counts()

plt.figure(figsize=(7, 7))
plt.pie(combo.values, labels=combo.index, autopct="%1.0f%%")
plt.show()

In [ ]:
# TODO: rediseña el gráfico deficiente 2 (o pega aquí la captura de Excel)


### Gráfico deficiente 3

(Es el mismo gráfico de doble eje de la Sección 7; vuelve a mostrarlo aquí como el "deficiente" a rediseñar.)

In [ ]:
tabla2.plot(secondary_y="edad_prom", style=["o-", "s-"])
plt.xticks(rotation=45)
plt.show()

In [ ]:
# TODO: rediseña el gráfico deficiente 3 (o pega aquí la captura de Excel)


**Gráfico 1, qué violaba y qué cambié:** *(eje truncado, color sin propósito, sin título ni unidades...)*

**Gráfico 2, qué violaba y qué cambié:** *(demasiadas categorías para un pastel...)*

**Gráfico 3, qué violaba y qué cambié:** *(doble eje con correlación real distinta a la que sugiere el gráfico...)*

## Cierre

Hoy construiste cada tipo de gráfico de la clase con datos reales: comparación, composición, distribución y relación; escalas honestas; color con propósito; y menos ruido visual. El criterio para los tres sigue siendo el mismo: **¿qué pregunta responde el gráfico, y se nota con solo mirarlo?**

Guarda una copia de este notebook en tu Google Drive antes de cerrar Colab.

**Próxima sesión, Clase 12 (MasterClass):** recapitulación presencial del Módulo 2 completo (Pandas, integración de fuentes y visualización) antes de entrar al Módulo 3 (Power BI).